<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week1/lesson2/Lesson2_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 2 — Exercises**
## Dot Products, Neural Similarity, Weighted Inputs, and Projection

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 1 — Linear Algebra I\
**Lesson:** Lesson 2

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Dot product calculation and meaning | 10 |
| 2 | Angles and orthogonality | 15 |
| 3 | Cosine similarity and neural response patterns | 15 |
| 4 | Projection and projection error | 15 |
| 5 | Weighted sums, bias, and a linear classifier | 15 |
| 6 | Neural task-axis mini-analysis | 15 |
| 7 | Real data: decoding reach direction from motor cortex | 15 |
| **Total** | | **100** |
| Bonus | Making two directions orthogonal (Gram–Schmidt) | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)   # exercise notebook uses its own fixed seed
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
print("Setup complete.")

---
## Exercise 1 — Dot Product Calculation and Meaning (10 points)

A neuron receives input from four presynaptic neurons:

- **Firing rates:** $\mathbf{x} = (15,\ 40,\ 8,\ 25)$ spikes/s
- **Connection weights:** $\mathbf{w} = (0.5,\ -0.25,\ 1.0,\ 0.2)$ (no units)

**1.1 (3 points, written response, no code).** Compute $z = \mathbf{w}\cdot\mathbf{x}$ by hand. Write out every product $w_i x_i$, then the total, with its unit.

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (2 points): Create x_in and w_in as NumPy arrays (values above, same order). Then compute
#   contributions = the four products w_i * x_i  (an array of shape (4,))
#   z_numpy       = the dot product, using the @ operator

# YOUR CODE HERE
x_in = None
w_in = None
contributions = None
z_numpy = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2 -- structural checks only, does not reveal the expected values.
assert x_in is not None and w_in is not None, "Define x_in and w_in."
assert np.asarray(contributions).shape == (4,), "contributions should have shape (4,)."
assert np.isscalar(z_numpy) or np.asarray(z_numpy).shape == (), "z_numpy should be a single number."
assert np.isclose(np.sum(contributions), z_numpy), "The contributions should add up to z_numpy."
print("Self-check passed: shapes look right and the contributions add up to z. (Compare z with your 1.1 answer.)")

In [ ]:
# 1.3 (3 points, debugging task): dot_loop() below is meant to compute the dot product with a
# Python loop, but one line is BUGGY. Find the bug and fix it (do not use np.dot or @ inside the loop).

def dot_loop(a, b):
    if a.shape != b.shape:
        raise ValueError(f"Shape mismatch: {a.shape} vs {b.shape}")
    total = 0.0
    for i in range(len(a)):
        total += a[i] * b[i - 1]     # YOUR FIX HERE
    return total

print("dot_loop(w_in, x_in) =", dot_loop(w_in, x_in))

In [ ]:
# SELF-CHECK 1.3 -- tests dot_loop on random vectors, not on the exercise values.
_test_rng = np.random.default_rng(0)
for _ in range(5):
    _a, _b = _test_rng.normal(size=6), _test_rng.normal(size=6)
    assert np.isclose(dot_loop(_a, _b), _a @ _b), "dot_loop still disagrees with NumPy's dot product."
print("Self-check passed: dot_loop agrees with NumPy on random test vectors.")

**1.4 (2 points, written response, conceptual).** Suppose someone stored the firing rates in a different neuron order from the weights, for example N2 first. Would NumPy raise an error? Explain what the resulting number would mean.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Angles and Orthogonality (15 points)

In [ ]:
# 2.1 (5 points): Complete angle_between(a, b) so that it returns the angle between a and b in DEGREES.
#   - Use cos(theta) = (a . b) / (||a|| ||b||).
#   - Clip the cosine to [-1, 1] with np.clip before calling np.arccos.
#   - Raise a ValueError if either vector is (numerically) the zero vector.

def angle_between(a, b, eps=1e-12):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1 -- general properties, not the exercise answers.
assert np.isclose(angle_between(np.array([1.0, 0.0]), np.array([0.0, 2.0])), 90.0), "Perpendicular vectors should give 90."
assert np.isclose(angle_between(np.array([1.0, 1.0]), np.array([-3.0, -3.0])), 180.0), "Opposite vectors should give 180."
_v = np.array([0.1, 0.2, 0.3])
assert not np.isnan(angle_between(_v, 3 * _v)), "Got nan for parallel vectors -- did you clip the cosine?"
try:
    angle_between(np.zeros(2), np.array([1.0, 0.0]))
    raise AssertionError("angle_between should raise ValueError for the zero vector.")
except ValueError:
    pass
print("Self-check passed: angle_between handles right angles, opposite and parallel vectors, and the zero vector.")

**2.2.** Three pairs of vectors are given below. Pair (b) is built from `cos` and `sin`, so its dot product will not come out as an exact zero even if the vectors are perpendicular on paper.

In [ ]:
pair_a = (np.array([2.0, 1.0]), np.array([1.5, 1.8]))
pair_b = (np.array([np.cos(np.pi / 3), np.sin(np.pi / 3)]),
          np.array([np.cos(np.pi / 3 + np.pi / 2), np.sin(np.pi / 3 + np.pi / 2)]))
pair_c = (np.array([1.0, 2.0]), np.array([-2.0, 0.5]))
pairs = [pair_a, pair_b, pair_c]

In [ ]:
# 2.2 (4 points): For each pair in `pairs`, compute the dot product and classify its sign as the string
# 'positive', 'zero', or 'negative'. Use np.isclose(..., 0.0, atol=1e-8) to decide 'zero'.
# Store the three dot products in pair_dots and the three labels in pair_signs (both lists, same order).

# YOUR CODE HERE
pair_dots = None
pair_signs = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert pair_dots is not None and len(pair_dots) == 3, "pair_dots should be a list of three numbers."
assert pair_signs is not None and len(pair_signs) == 3, "pair_signs should be a list of three labels."
assert all(s in ('positive', 'zero', 'negative') for s in pair_signs), "Use only 'positive', 'zero', or 'negative'."
print("Self-check passed: three dot products and three valid labels.")

**2.3.** Two downstream neurons read out the same four-neuron population with weight vectors

$$\mathbf{w}_1 = (1,\ 2,\ 0,\ -1), \qquad \mathbf{w}_2 = (2,\ -1,\ 3,\ 0)$$

and the population then changes by $\Delta\mathbf{r} = 3\,\mathbf{w}_1$ spikes/s (a change *along* $\mathbf{w}_1$).

In [ ]:
# 2.3 (3 points): Using w1 and w2 below,
#   readouts_orthogonal = True/False: are w1 and w2 orthogonal? (use np.isclose)
#   change_1 = change in read-out 1 caused by delta_r   (i.e. w1 . delta_r)
#   change_2 = change in read-out 2 caused by delta_r
w1 = np.array([1.0, 2.0, 0.0, -1.0])
w2 = np.array([2.0, -1.0, 3.0, 0.0])
delta_r = 3.0 * w1

# YOUR CODE HERE
readouts_orthogonal = None
change_1 = None
change_2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.3
assert readouts_orthogonal in (True, False), "readouts_orthogonal should be True or False."
assert change_1 is not None and change_2 is not None, "Compute change_1 and change_2."
assert np.isclose(change_1, 3 * (w1 @ w1)), "change_1 should be w1 . delta_r."
print("Self-check passed. (Now explain change_2 in 2.4.)")

**2.4 (3 points, written response).** (a) Explain geometrically why `change_2` came out as it did. (b) In pair (b) of 2.2, the dot product printed as a tiny number rather than `0.0`. Why, and why is `np.isclose` a better test than `== 0`?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Cosine Similarity and Neural Response Patterns (15 points)

Four auditory neurons are recorded while five sounds are played. Each sound gives one population vector (mean firing rate, spikes/s):

In [ ]:
neuron_labels = ['N1', 'N2', 'N3', 'N4']
tone_A = np.array([30.0, 12.0, 4.0, 6.0])
sound_responses = {
    'Tone A':        tone_A,
    'Tone A (loud)': 2.5 * tone_A,
    'Tone B':        np.array([5.0, 8.0, 28.0, 20.0]),
    'Tones A + B':   np.array([18.0, 11.0, 16.0, 13.0]),
    'Silence':       np.array([6.0, 7.0, 6.0, 6.0]),
}
sound_names = list(sound_responses.keys())
responses = np.stack(list(sound_responses.values()))   # shape (n_sounds, n_neurons) = (5, 4), spikes/s

def normalise(vec, eps=1e-12):
    # Provided for you (from Lesson 1): unit vector of vec, or ValueError for the zero vector.
    vec = np.asarray(vec, dtype=float)
    n = np.linalg.norm(vec)
    if n < eps:
        raise ValueError("The zero vector has no defined direction.")
    return vec / n

print("responses shape:", responses.shape, "(n_sounds, n_neurons)")

In [ ]:
# 3.1 (4 points): Complete cosine_similarity(a, b) using normalise() and a dot product.
# Clip the result to [-1, 1]. A zero vector should raise a ValueError (normalise already does this).

def cosine_similarity(a, b):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1 -- general properties only.
_v = np.array([1.0, 2.0, 3.0])
assert np.isclose(cosine_similarity(_v, 5 * _v), 1.0), "A vector and a positive multiple should give 1."
assert np.isclose(cosine_similarity(_v, -_v), -1.0), "A vector and its negative should give -1."
assert np.isclose(cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 4.0])), 0.0), "Orthogonal vectors should give 0."
try:
    cosine_similarity(_v, np.zeros(3))
    raise AssertionError("cosine_similarity should raise ValueError for the zero vector.")
except ValueError:
    pass
print("Self-check passed: cosine_similarity behaves correctly on the test cases.")

In [ ]:
# 3.2 (4 points): Build sim_matrix, a (5, 5) array where sim_matrix[i, j] is the cosine similarity
# between responses[i] and responses[j]. Then print it rounded to 2 decimals.

# YOUR CODE HERE
sim_matrix = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert sim_matrix is not None and np.asarray(sim_matrix).shape == (5, 5), "sim_matrix should have shape (5, 5)."
assert np.allclose(np.diag(sim_matrix), 1.0), "Every pattern should have similarity 1 with itself."
assert np.allclose(sim_matrix, sim_matrix.T), "sim_matrix should be symmetric."
print("Self-check passed: shape, diagonal, and symmetry are right.")

In [ ]:
# 3.3 (3 points): Compare 'Tone A' with 'Silence' in two ways:
#   sim_raw      = cosine similarity of the raw vectors
#   sim_centred  = cosine similarity after subtracting each vector's own mean (mean-centring)

# YOUR CODE HERE
sim_raw = None
sim_centred = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.3
assert sim_raw is not None and sim_centred is not None, "Compute both similarities."
assert -1.0 <= sim_raw <= 1.0 and -1.0 <= sim_centred <= 1.0, "Cosine similarities must lie in [-1, 1]."
print("Self-check passed.")

**3.4 (4 points, written response, scientific interpretation).** Using your matrix and 3.3: (a) Which pair of *different* sounds is most similar, and what does that tell you about the two responses? (b) Silence is almost a flat pattern, yet its raw similarity to Tone A is high. Why? (c) What did mean-centring change, and why should you report it when you use it?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Projection and Projection Error (15 points)

Use $\mathbf{u} = (2,\ 5)$ and the direction $\mathbf{v} = (3,\ 1)$. Note that $\mathbf{v}$ is **not** a unit vector.

In [ ]:
# 4.1 (5 points): Complete vector_projection(u, v) using  proj_v(u) = (u . v) / (v . v) * v,
# raising a ValueError if v is (numerically) the zero vector. Then compute
#   comp_u = the scalar projection of u onto v   (a single number)
#   proj_u = the vector projection of u onto v   (shape (2,))
u = np.array([2.0, 5.0])
v = np.array([3.0, 1.0])

def vector_projection(u, v, eps=1e-12):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE
comp_u = None
proj_u = None
raise NotImplementedError

In [ ]:
# 4.2 (3 points): Compute the residual e = u - proj_u and the projection error ||e||.

# YOUR CODE HERE
residual = None
projection_error = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1-4.2 -- checks the defining properties of a projection, not the values.
assert proj_u is not None and residual is not None, "Compute proj_u and residual."
assert np.allclose(proj_u + residual, u), "proj_u + residual should rebuild u."
assert np.isclose(residual @ v, 0.0), "The residual should be orthogonal to v."
assert np.isclose(proj_u[0] * v[1] - proj_u[1] * v[0], 0.0), "proj_u should point along v."
assert np.isclose(abs(comp_u), np.linalg.norm(proj_u)), "|comp_u| should equal the length of proj_u."
try:
    vector_projection(u, np.zeros(2))
    raise AssertionError("vector_projection should raise ValueError for a zero v.")
except ValueError:
    pass
print("Self-check passed: proj_u lies along v, the residual is orthogonal to v, and together they rebuild u.")

In [ ]:
# 4.3 (4 points): Draw u, v, and proj_u as arrows from the origin (plt.quiver), and the residual as a
# dashed line from the tip of proj_u to the tip of u. Label both axes, add a title and a legend,
# and make the aspect ratio equal.

# YOUR CODE HERE
raise NotImplementedError

**4.4 (3 points, written response, conceptual).** In your own words: what part of $\mathbf{u}$ does the projection keep, and what does the residual contain? Why is the projection error the *shortest* distance from $\mathbf{u}$ to the line through $\mathbf{v}$? (Hint: think about the right angle in your figure.)

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Weighted Sums, Bias, and a Linear Classifier (15 points)

In [ ]:
# 5.1 (4 points): A model neuron has five inputs, five weights, and a bias. Compute
#   contributions5 = the five products w_i * x_i   (shape (5,))
#   z5             = w . x + b                      (a single number, spikes/s)
x5 = np.array([10.0, 25.0, 0.0, 14.0, 6.0])     # presynaptic rates, spikes/s
w5 = np.array([0.9, -0.3, 0.5, 0.4, 0.0])       # weights, no units
b5 = -2.0                                       # bias, spikes/s

# YOUR CODE HERE
contributions5 = None
z5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert np.asarray(contributions5).shape == (5,), "contributions5 should have shape (5,)."
assert np.isclose(np.sum(contributions5) + b5, z5), "z5 should equal the sum of contributions plus the bias."
print("Self-check passed.")

**5.2 (3 points, written response).** Which inputs raise $z$ and which lower it? Input 3 has a nonzero weight and input 5 has a nonzero rate, yet both contribute nothing. Why? What does the bias do that no input can?

**Your answer:**

*(replace this line with your answer)*

**5.3.** A linear classifier uses the same computation. Eight observations with two (standardised) features each are given below, with their true class labels. The classifier predicts class 1 when $z = \mathbf{w}\cdot\mathbf{x} + b > 0$.

In [ ]:
X_obs = np.array([[ 1.2, -0.4], [ 0.8,  0.3], [ 2.0,  1.1], [ 0.1, -1.5],
                  [-1.0,  0.2], [-0.5,  1.4], [-1.8, -0.6], [ 0.2,  1.0]])   # shape (8, 2): (observation, feature)
y_obs = np.array([1, 1, 1, 1, 0, 0, 0, 0])                                    # true labels, shape (8,)
w_c = np.array([1.5, -1.0])
b_c = 0.5

In [ ]:
# 5.3 (4 points): Compute
#   scores      = one score z for each observation   (shape (8,); a single line with @ is enough)
#   predictions = 1 where the score is > 0, else 0    (integers, shape (8,))
#   accuracy    = fraction of predictions equal to y_obs

# YOUR CODE HERE
scores = None
predictions = None
accuracy = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.3
assert np.asarray(scores).shape == (8,), "scores should have shape (8,)."
assert np.asarray(predictions).shape == (8,) and set(np.unique(predictions)) <= {0, 1}, "predictions should be 0s and 1s, shape (8,)."
assert np.all((np.asarray(scores) > 0) == (np.asarray(predictions) == 1)), "predictions should be 1 exactly where the score is positive."
assert 0.0 <= accuracy <= 1.0, "accuracy should be a fraction between 0 and 1."
print("Self-check passed.")

**5.4 (4 points, written response, conceptual).** (a) How is the decision boundary $\mathbf{w}\cdot\mathbf{x} + b = 0$ oriented relative to $\mathbf{w}$? (b) If you changed `b_c` from `0.5` to `2.5`, would the boundary turn, or move? Which class would be predicted more often, and why?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Neural Task-Axis Mini-Analysis (15 points)

The cell below generates a reproducible recording for you (do not modify it). Three neurons are recorded while an animal reaches to the **left** or the **right**: 30 training trials and 30 test trials per direction.

In [ ]:
# PROVIDED -- do not modify.
n_trials = 30
mean_left_true = np.array([22.0, 10.0, 15.0])    # spikes/s, (N1, N2, N3)
mean_right_true = np.array([12.0, 20.0, 15.0])
noise_sd = 4.0

train_left = mean_left_true + RNG.normal(0.0, noise_sd, size=(n_trials, 3))    # shape (n_trials, n_neurons) = (30, 3)
train_right = mean_right_true + RNG.normal(0.0, noise_sd, size=(n_trials, 3))
test_left = mean_left_true + RNG.normal(0.0, noise_sd, size=(n_trials, 3))
test_right = mean_right_true + RNG.normal(0.0, noise_sd, size=(n_trials, 3))

print("train_left shape:", train_left.shape, " (n_trials, n_neurons), units: spikes/s")

In [ ]:
# 6.1 (3 points): From the TRAINING trials only, compute
#   mean_left, mean_right = the mean population response for each direction   (shape (3,) each)
#   task_axis             = the unit vector pointing from mean_right to mean_left

# YOUR CODE HERE
mean_left = None
mean_right = None
task_axis = None
raise NotImplementedError

In [ ]:
# 6.2 (3 points): Project every TRAINING trial onto task_axis:
#   coords_left, coords_right = one coordinate per trial   (shape (30,) each)

# YOUR CODE HERE
coords_left = None
coords_right = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1-6.2
for name, val, shape in [("mean_left", mean_left, (3,)), ("mean_right", mean_right, (3,)),
                         ("task_axis", task_axis, (3,)), ("coords_left", coords_left, (30,)),
                         ("coords_right", coords_right, (30,))]:
    assert val is not None, f"{name} is still None."
    assert np.asarray(val).shape == shape, f"{name} should have shape {shape}."
assert np.isclose(np.linalg.norm(task_axis), 1.0), "task_axis should be a unit vector."
assert np.mean(coords_left) > np.mean(coords_right), "task_axis should point towards the left condition."
print("Self-check passed.")

In [ ]:
# 6.3 (3 points): Plot overlapping histograms of coords_left and coords_right (use the same bins for both).
# Label both axes with units, add a title and a legend.

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# 6.4 (3 points): Turn the axis into a decoder z = task_axis . r + b_dec that says "left" when z > 0.
#   b_dec         = -(midpoint . task_axis), where midpoint is halfway between mean_left and mean_right
#   test_accuracy = fraction of the 60 TEST trials decoded correctly

# YOUR CODE HERE
b_dec = None
test_accuracy = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.4
assert b_dec is not None and test_accuracy is not None, "Compute b_dec and test_accuracy."
assert 0.0 <= test_accuracy <= 1.0, "test_accuracy should be a fraction between 0 and 1."
assert test_accuracy > 0.5, "A decoder pointing the right way should beat chance (50%). Check the sign of task_axis or b_dec."
print("Self-check passed.")

**6.5 (3 points, written response — interpretation and limitation).** Look at the three entries of `task_axis`. Which neurons carry information about reach direction, and which does not? Compare the test accuracy to chance. Give one limitation of this analysis.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Decoding Reach Direction from Motor Cortex (15 points)

This exercise uses the real recording from Part 12 of the teaching notebook: 171 neurons in a monkey's primary motor cortex while it made centre-out reaches to eight targets ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); shared through the [DREAM database on CRCNS](https://crcns.org/data-sets/movements/dream) and packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)).

You will fit each neuron's cosine tuning curve with dot products, then use the whole population to **decode** which way the hand was moving, as Georgopoulos et al. (1986) did. To make the test honest, the recording is split in two:
- the **first half** of the session is used to fit the tuning curves (`tuning_fit`);
- the **second half** is used only to test the decoder (`tuning_test`).

The cell below downloads the file (1.3 MB) the first time it runs, and builds both tables. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
m1 = loadmat(M1_FILE)
rates = m1['spikes'].astype(float) / 0.05            # shape (171, 15536): (neuron, 50 ms time bin), spikes/s
hand_vel = m1['handVel'][:2]                          # shape (2, 15536): x and y hand velocity, m/s

angles_deg = np.arange(0, 360, 45)                    # shape (8,): the eight reach directions, degrees
direction_index = np.round((np.degrees(np.arctan2(hand_vel[1], hand_vel[0])) % 360) / 45).astype(int) % 8
moving = np.linalg.norm(hand_vel, axis=0) > 0.1       # bins where the hand moves faster than 10 cm/s
first_half = np.arange(rates.shape[1]) < rates.shape[1] // 2

def tuning_table(use):
    # Mean rate of every neuron in each of the 8 directions, using only the bins where `use` is True.
    return np.stack([rates[:, use & moving & (direction_index == k)].mean(axis=1) for k in range(8)], axis=1)

tuning_fit = tuning_table(first_half)                 # shape (171, 8): (neuron, direction), spikes/s -- FIT data
tuning_test = tuning_table(~first_half)               # shape (171, 8): (neuron, direction), spikes/s -- TEST data
print("tuning_fit:", tuning_fit.shape, " tuning_test:", tuning_test.shape, " (n_neurons, n_directions), spikes/s")
print("Row i is neuron i's tuning curve, a vector of 8 rates; column k is direction angles_deg[k].")

In [ ]:
# 7.1 (4 points): Write cosine_fit(curve) for a tuning curve of 8 rates (spikes/s), one per entry of angles_deg.
# Following Part 12 of the teaching notebook, it returns (b, g, pd, similarity):
#   b          -- the baseline, the mean of the curve (spikes/s)
#   g          -- the depth of modulation, ||p|| / 4 (spikes/s), where p = (c . cos_vec, c . sin_vec),
#                 c is the centred curve, and cos_vec, sin_vec hold the cos and sin of the 8 angles
#   pd         -- the preferred direction in degrees, between 0 and 360 (use np.arctan2 on p)
#   similarity -- the cosine similarity between c and the centred model g * cos(angles - pd)
# If the centred curve is all zeros, return (b, 0.0, np.nan, 0.0).
# Then apply it to every row of tuning_fit, to make fit_params with shape (171, 4).

def cosine_fit(curve):
    # YOUR CODE HERE
    raise NotImplementedError

fit_params = None   # replace: shape (171, 4), one row (b, g, pd, similarity) per neuron

In [ ]:
# SELF-CHECK 7.1 -- tested on made-up curves, so it does not reveal the real values.
assert fit_params is not None, "fit_params is still None."
assert np.asarray(fit_params).shape == (171, 4), "fit_params should have one row (b, g, pd, similarity) per neuron: shape (171, 4)."
test_curve = 20 + 8 * np.cos(np.radians(angles_deg - 200.0))
b_t, g_t, pd_t, s_t = cosine_fit(test_curve)
assert np.isclose(b_t, 20) and np.isclose(g_t, 8), "On a perfect cosine with baseline 20 and depth 8, cosine_fit should return b = 20 and g = 8."
assert np.isclose(pd_t, 200.0), "The preferred direction should be in degrees, between 0 and 360."
assert np.isclose(s_t, 1.0), "A perfect cosine should have similarity 1."
flat = cosine_fit(np.full(8, 5.0))
assert flat[1] == 0.0 and flat[3] == 0.0, "A flat curve should give g = 0 and similarity 0."
print("Self-check passed.")

In [ ]:
# 7.2 (5 points): The population vector. For a set of neurons with fitted b, g and pd (degrees) and observed rates r
# (all vectors of the same length), weight each neuron by how far its rate is above baseline, in units of its depth:
#     w_i = (r_i - b_i) / g_i
# and add up the neurons' preferred-direction unit vectors with these weights. The x and y components of the sum
# are two dot products:   pv_x = w . cos(pd),   pv_y = w . sin(pd).
# Write decode_direction(r, b, g, pd) that returns the angle of (pv_x, pv_y) in degrees, between 0 and 360.
#
# Then decode the TEST data using only the well-tuned neurons (sim_fit > 0.8):
#   decoded -- shape (8,): the decoded angle for each column of tuning_test
#   errors  -- shape (8,): decoded minus true angle, wrapped into [-180, 180) with (d + 180) % 360 - 180

def decode_direction(r, b, g, pd):
    # YOUR CODE HERE
    raise NotImplementedError

decoded = None   # replace
errors = None    # replace

In [ ]:
# SELF-CHECK 7.2 -- the function is tested on a made-up population, so the real values stay hidden.
assert decoded is not None and errors is not None, "decoded and errors are still None."
assert np.asarray(decoded).shape == (8,) and np.asarray(errors).shape == (8,), "decoded and errors need one value per direction: shape (8,)."
assert np.all((np.asarray(decoded) >= 0) & (np.asarray(decoded) < 360)), "Decoded angles should lie between 0 and 360 degrees."
assert np.all((np.asarray(errors) >= -180) & (np.asarray(errors) < 180)), "Wrap the errors into [-180, 180)."
pd_toy = np.arange(0, 360, 30.0)                                  # 12 made-up neurons, evenly spread
b_toy, g_toy = np.full(12, 30.0), np.full(12, 10.0)
r_toy = b_toy + g_toy * np.cos(np.radians(130.0 - pd_toy))        # their rates for a 130-degree movement
assert np.isclose(decode_direction(r_toy, b_toy, g_toy, pd_toy), 130.0), "A perfect, evenly spread population should decode 130 degrees exactly."
print("Self-check passed.")

In [ ]:
# 7.3 (2 points): Repeat the decoding with the RAW test rates as the weights (w_i = r_i: no baseline subtracted,
# no division by g). Store the wrapped errors in errors_raw, shape (8,), and print the mean absolute error of both decoders.
# Hint: decode_direction(r, 0, 1, pd) uses exactly these weights.

# YOUR CODE HERE
errors_raw = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert errors_raw is not None, "errors_raw is still None."
assert np.asarray(errors_raw).shape == (8,), "errors_raw needs one value per direction: shape (8,)."
assert np.all((np.asarray(errors_raw) >= -180) & (np.asarray(errors_raw) < 180)), "Wrap errors_raw into [-180, 180)."
print("Self-check passed.")

**7.4 (4 points, written response — interpretation and limitation).** How well does the population vector decode reach direction on data it was not fitted to? Why does using the raw rates as weights fail so badly? (Think about what all the weights have in common, and look back at Part 6 of the teaching notebook.) Give one limitation of this analysis.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Making Two Directions Orthogonal (extra credit, not part of the 100 points)

Two population directions $\mathbf{d}_1 = (2,\ 1,\ 0)$ and $\mathbf{d}_2 = (1,\ 3,\ 1)$ are not orthogonal. Subtracting from $\mathbf{d}_2$ its projection onto $\mathbf{d}_1$ leaves a direction $\mathbf{q}_2$ that *is* orthogonal to $\mathbf{d}_1$ (the first step of the **Gram–Schmidt** procedure).

1. Compute $\mathbf{q}_2 = \mathbf{d}_2 - \operatorname{proj}_{\mathbf{d}_1}(\mathbf{d}_2)$ using your `vector_projection` from Exercise 4.
2. Verify that $\mathbf{q}_2$ is orthogonal to $\mathbf{d}_1$.
3. In writing, explain why removing the projection always produces an orthogonal vector, and suggest one reason a neuroscientist might want to remove a direction (for example, an overall "arousal" direction) from population data.

**Your code:**

In [ ]:
d1 = np.array([2.0, 1.0, 0.0])
d2 = np.array([1.0, 3.0, 1.0])

# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.